<a href="https://colab.research.google.com/github/Rzan-AlSaadoun/News-Classification/blob/main/NewsClassificatoinASG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#installing libraries
!pip install -q scikit-learn pandas numpy matplotlib seaborn nltk joblib gradio

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Data Loading and Exploration


In [ ]:
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

print("COMBINED DATA LOADING AND EXPLORATION")
print("=" * 50)

def extract_all_datasets():
    """Extract both NADCG and SANAD datasets"""
    datasets = [
        "/content/drive/MyDrive/NLP Assignment datasets/NADCG New Arabic dataset for text classification and generation.zip",
        "/content/drive/MyDrive/NLP Assignment datasets/SANAD_SUBSET.zip"
    ]

    for dataset_zip in datasets:
        zip_name = Path(dataset_zip).stem
        extract_to = Path("/content/data") / zip_name
        extract_to.mkdir(parents=True, exist_ok=True)

        print(f"Extracting {zip_name}...")
        with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"{zip_name} extracted!")

extract_all_datasets()

def load_combined_datasets():
    """Load both NADCG and SANAD datasets and combine them"""
    rows = []

    # Load NADCG dataset
    print("LOADING NADCG DATASET...")
    nadcg_folder = "/content/data/NADCG New Arabic dataset for text classification and generation"
    csv_files = list(Path(nadcg_folder).rglob("*.csv"))

    if csv_files:
        nadcg_path = csv_files[0]
        df_nadcg = pd.read_csv(nadcg_path, encoding='utf-8')
        print(f"NADCG shape: {df_nadcg.shape}")

        for idx, row in df_nadcg.iterrows():
            category_text = row['output']
            # Fix NADCG category format
            if isinstance(category_text, str) and category_text.startswith('category:'):
                parts = category_text.split()
                if len(parts) >= 2:
                    category_num = parts[1].replace('article:', '')
                    category = f"NADCG_Cat_{category_num}"
                else:
                    category = "NADCG_Unknown"
            else:
                category = f"NADCG_{category_text}"

            rows.append({
                "source": "NADCG",
                "original_category": category,
                "text": row['input'],
                "filename": "NADCG.csv"
            })

    # Load SANAD dataset - handle all sub-datasets
    print("LOADING SANAD DATASET...")
    sanad_base_path = Path("/content/data/SANAD_SUBSET")

    # SANAD contains multiple datasets: Akhbarona, Arabiya, Khaleej, etc.
    sanad_sources = ['Akhbarona', 'Arabiya', 'Khaleej', 'Train', 'Test']

    total_sanad_count = 0
    for source in sanad_sources:
        source_count = 0
        source_path = sanad_base_path / source

        if source_path.exists():
            # Find all text files in this source
            txt_files = list(source_path.rglob("*.txt"))

            for txt_file in txt_files:
                try:
                    # Category is the parent folder name
                    category = txt_file.parent.name
                    with open(txt_file, "r", encoding="utf-8", errors="ignore") as f:
                        text = f.read().strip()

                    if text:  # Only add non-empty texts
                        rows.append({
                            "source": f"SANAD_{source}",
                            "original_category": category,
                            "text": text,
                            "filename": txt_file.name
                        })
                        source_count += 1
                        total_sanad_count += 1

                except Exception as e:
                    print(f"Error reading {txt_file}: {e}")
                    continue

            print(f"  {source}: {source_count} articles")
        else:
            print(f"  {source}: directory not found")

    print(f"Total SANAD articles loaded: {total_sanad_count}")

    return pd.DataFrame(rows)

# Load combined data
df_combined = load_combined_datasets()
print(f"COMBINED DATASET CREATED!")
print(f"Total articles: {len(df_combined)}")
print(f"Sources: {df_combined['source'].value_counts().to_dict()}")

# Display dataset structure
print(f"DATASET STRUCTURE:")
print(f"Columns: {df_combined.columns.tolist()}")
print(f"Sample categories from NADCG: {df_combined[df_combined['source'] == 'NADCG']['original_category'].unique()[:5]}")
print(f"Sample categories from SANAD: {df_combined[df_combined['source'].str.startswith('SANAD')]['original_category'].unique()[:10]}")


COMBINED DATA LOADING AND EXPLORATION
Extracting NADCG New Arabic dataset for text classification and generation...
NADCG New Arabic dataset for text classification and generation extracted!
Extracting SANAD_SUBSET...
SANAD_SUBSET extracted!
LOADING NADCG DATASET...
NADCG shape: (4000, 2)
LOADING SANAD DATASET...
  Akhbarona: directory not found
  Arabiya: directory not found
  Khaleej: directory not found
  Train: directory not found
  Test: directory not found
Total SANAD articles loaded: 0
COMBINED DATASET CREATED!
Total articles: 4000
Sources: {'NADCG': 4000}
DATASET STRUCTURE:
Columns: ['source', 'original_category', 'text', 'filename']
Sample categories from NADCG: ['NADCG_Cat_2' 'NADCG_Cat_5' 'NADCG_Cat_1' 'NADCG_Cat_6' 'NADCG_Cat_8']
Sample categories from SANAD: []


In [ ]:
#Category Mapping for Combined Dataset
print("CATEGORY MAPPING FOR COMBINED DATASET")
print("=" * 50)

# Comprehensive category mapping for both datasets
unified_category_mapping = {
    # NADCG categories
    'NADCG_Cat_1': 'Politics',
    'NADCG_Cat_2': 'Economy',
    'NADCG_Cat_3': 'Technology',
    'NADCG_Cat_4': 'Health',
    'NADCG_Cat_5': 'Culture',
    'NADCG_Cat_6': 'Sports',
    'NADCG_Cat_7': 'Religion',
    'NADCG_Cat_8': 'Other',
    'NADCG_1': 'Politics',
    'NADCG_2': 'Economy',
    'NADCG_3': 'Technology',
    'NADCG_4': 'Health',
    'NADCG_5': 'Culture',
    'NADCG_6': 'Sports',
    'NADCG_7': 'Religion',
    'NADCG_8': 'Other',

    # SANAD categories from all sub-datasets
    'culture': 'Culture',
    'politics': 'Politics',
    'sports': 'Sports',
    'tech': 'Technology',
    'finance': 'Economy',
    'medical': 'Health',
    'religion': 'Religion',
    'Culture': 'Culture',
    'Politics': 'Politics',
    'Sports': 'Sports',
    'Tech': 'Technology',
    'Finance': 'Economy',
    'Medical': 'Health',
    'Religion': 'Religion',

    # Common Arabic category names
    'اقتصاد': 'Economy',
    'رياضة': 'Sports',
    'سياسة': 'Politics',
    'ثقافة': 'Culture',
    'تكنولوجيا': 'Technology',
    'صحة': 'Health',
    'دين': 'Religion'
}

print("COMPREHENSIVE CATEGORY MAPPING:")
# Group mappings by target category for better display
reverse_mapping = {}
for old_cat, new_cat in unified_category_mapping.items():
    if new_cat not in reverse_mapping:
        reverse_mapping[new_cat] = []
    reverse_mapping[new_cat].append(old_cat)

for new_cat, old_cats in sorted(reverse_mapping.items()):
    print(f"{new_cat}: {old_cats}")

# Apply unified mapping
df_combined['unified_category'] = df_combined['original_category'].map(unified_category_mapping)

# Identify unmapped categories
unmapped = df_combined[df_combined['unified_category'].isna()]
if len(unmapped) > 0:
    print(f"UNMAPPED CATEGORIES ({len(unmapped)} articles):")
    unmapped_counts = unmapped['original_category'].value_counts()
    print(unmapped_counts.head(10))  # Show top 10 unmapped categories

    # Assign unmapped categories to 'Other'
    df_combined['unified_category'] = df_combined['unified_category'].fillna('Other')
    print(f"Assigned {len(unmapped)} articles with unmapped categories to 'Other'")
else:
    print("All categories successfully mapped!")

print(f"\nCATEGORY DISTRIBUTION AFTER UNIFICATION:")
category_stats = df_combined['unified_category'].value_counts()
print(category_stats)

print(f"\nCOMBINED DATASET SUMMARY:")
print(f"Total articles: {len(df_combined)}")
print(f"Unified categories: {df_combined['unified_category'].nunique()}")
print(f"Original categories: {df_combined['original_category'].nunique()}")
print(f"Sources: {df_combined['source'].value_counts().to_dict()}")
print(f"Articles per unified category:")
for category, count in category_stats.items():
    print(f"  {category}: {count} articles")

# Update main dataframe
df = df_combined
print(f"Combined dataset ready for preprocessing!")

CATEGORY MAPPING FOR COMBINED DATASET
COMPREHENSIVE CATEGORY MAPPING:
Culture: ['NADCG_Cat_5', 'NADCG_5', 'culture', 'Culture', 'ثقافة']
Economy: ['NADCG_Cat_2', 'NADCG_2', 'finance', 'Finance', 'اقتصاد']
Health: ['NADCG_Cat_4', 'NADCG_4', 'medical', 'Medical', 'صحة']
Other: ['NADCG_Cat_8', 'NADCG_8']
Politics: ['NADCG_Cat_1', 'NADCG_1', 'politics', 'Politics', 'سياسة']
Religion: ['NADCG_Cat_7', 'NADCG_7', 'religion', 'Religion', 'دين']
Sports: ['NADCG_Cat_6', 'NADCG_6', 'sports', 'Sports', 'رياضة']
Technology: ['NADCG_Cat_3', 'NADCG_3', 'tech', 'Tech', 'تكنولوجيا']
All categories successfully mapped!

CATEGORY DISTRIBUTION AFTER UNIFICATION:
unified_category
Health        543
Economy       538
Politics      513
Culture       507
Sports        485
Religion      476
Technology    472
Other         466
Name: count, dtype: int64

COMBINED DATASET SUMMARY:
Total articles: 4000
Unified categories: 8
Original categories: 8
Sources: {'NADCG': 4000}
Articles per unified category:
  Health: 543

#Text Preprocessing

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

print("TEXT PREPROCESSING")
print("=" * 50)

# Downloading Arabic stopwords
nltk.download('stopwords')
arabic_sw = set(stopwords.words('arabic'))

print(f"Loaded {len(arabic_sw)} Arabic stopwords")

# Defining Arabic text normalisation function
def normalise_arabic(text):
    if not isinstance(text, str) or pd.isna(text):  # ADD NULL CHECK
        return ""

    # Removing diacritics
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    # Normalising Alef variants to standard Alef
    text = re.sub('[إأآا]', 'ا', text)
    # Normalising ة to ه
    text = re.sub(r'ة', 'ه', text)
    # Normalising ى to ي
    text = re.sub(r'ى', 'ي', text)
    # Removing diacritics (الحركات)
    text = re.sub(r'ـ', '', text)
    # Removing non-Arabic characters, except spaces
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    # Removing extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Define tokenization function
def tokenize_arabic(text):
    """Tokenizes Arabic text and removes stopwords"""
    # Normalise first
    normalised_text = normalise_arabic(text)
    if not normalised_text: #empty check
        return []
    # Whitespace tokenization
    tokens = normalised_text.split()
    # Removing stopwords
    tokens = [token for token in tokens if token not in arabic_sw]
    return tokens

# Test preprocessing on sample articles
print("Testing preprocessing on sample articles:")
print("=" * 50)

# Clean data first
df['text'] = df['text'].fillna('')

# Testing for first 3 articles
for i in range(3):
    original_text = df['text'].iloc[i]
    normalised_text = normalise_arabic(original_text)
    tokens = tokenize_arabic(original_text)

    print(f"\nSample {i+1}:")
    print(f"Original ({len(original_text)} chars): {original_text[:100]}...")
    print(f"Normalised ({len(normalised_text)} chars): {normalised_text[:100]}...")
    print(f"Tokens ({len(tokens)} words): {tokens[:10]}...")
    print("-" * 30)

# Apply preprocessing to entire dataset
print("\nApplying preprocessing to entire dataset...")
df['normalised_text'] = df['text'].apply(normalise_arabic)
df['tokens'] = df['text'].apply(tokenize_arabic)
df['token_count'] = df['tokens'].str.len()

print("Preprocessing completed!")
print(f"Average tokens per article: {df['token_count'].mean():.1f}")

TEXT PREPROCESSING
Loaded 701 Arabic stopwords
Testing preprocessing on sample articles:

Sample 1:
Original (56 chars): البورصه المصريه تخسر مليار في دقاءق ومءشرها يتراجع بنسبه...
Normalised (56 chars): البورصه المصريه تخسر مليار في دقاءق ومءشرها يتراجع بنسبه...
Tokens (8 words): ['البورصه', 'المصريه', 'تخسر', 'مليار', 'دقاءق', 'ومءشرها', 'يتراجع', 'بنسبه']...
------------------------------

Sample 2:
Original (50 chars): وكالهالطاقه الدوليه الطلب العالمي الفحم يظل مستقرا...
Normalised (50 chars): وكالهالطاقه الدوليه الطلب العالمي الفحم يظل مستقرا...
Tokens (7 words): ['وكالهالطاقه', 'الدوليه', 'الطلب', 'العالمي', 'الفحم', 'يظل', 'مستقرا']...
------------------------------

Sample 3:
Original (55 chars): بيت القصيد ينظم امسيه شعريه لغاده الشريف ومصطفي عبد ربه...
Normalised (55 chars): بيت القصيد ينظم امسيه شعريه لغاده الشريف ومصطفي عبد ربه...
Tokens (10 words): ['بيت', 'القصيد', 'ينظم', 'امسيه', 'شعريه', 'لغاده', 'الشريف', 'ومصطفي', 'عبد', 'ربه']...
------------------------------

App

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Preprocessing completed!
Average tokens per article: 8.6


#Feature Engineering and Vectorisation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from scipy.sparse import hstack

print("FEATURE ENGINEERING")
print("=" * 50)


# Use balanced sample for efficiency
sample_size = min(10000, len(df))
df_balanced = df.sample(sample_size, random_state=42)
print(f"Using balanced sample of {len(df_balanced)} articles")

# Add exactly two hand-crafted features as required
print("Adding hand-crafted features...")
df_balanced['article_length'] = df_balanced['normalised_text'].str.len()
df_balanced['unique_word_count'] = df_balanced['tokens'].apply(lambda x: len(set(x)))

print("Hand-crafted features added:")
print("- Article length (character count)")
print("- Unique word count")

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_tfidf = vectorizer.fit_transform(df_balanced['normalised_text'])
print(f"\nTF-IDF features shape: {X_tfidf.shape}")

# Select top features using chi-square as required
print("Selecting top features using chi-square...")
k_best = 2000
selector = SelectKBest(chi2, k=k_best)
X_tfidf_selected = selector.fit_transform(X_tfidf, df_balanced['unified_category'])

print(f"Selected {X_tfidf_selected.shape[1]} top features using chi-square")

# Combine TF-IDF with hand-crafted features
handcrafted_features = df_balanced[['article_length', 'unique_word_count']].values
X_combined = hstack([X_tfidf_selected, handcrafted_features])
y = df_balanced['unified_category']

print(f"Final feature matrix shape: {X_combined.shape}")
print("Feature engineering completed!")

FEATURE ENGINEERING
Using balanced sample of 4000 articles
Adding hand-crafted features...
Hand-crafted features added:
- Article length (character count)
- Unique word count

TF-IDF features shape: (4000, 5000)
Selecting top features using chi-square...
Selected 2000 top features using chi-square
Final feature matrix shape: (4000, 2002)
Feature engineering completed!


#Model Training and Evaluation


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

print("MODEL TRAINING AND EVALUATION")
print("=" * 50)

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} articles")
print(f"Test set: {X_test.shape[0]} articles")
print(f"Number of features: {X_train.shape[1]}")

# Initialize three classifiers
print("\nInitializing classifiers...")
classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': LinearSVC(random_state=42),
    'Naive Bayes': MultinomialNB()
}

# Train models
print("Training models...")
for name, classifier in classifiers.items():
    classifier.fit(X_train, y_train)
    print(f"  {name} trained")

# Make predictions
predictions = {}
for name, classifier in classifiers.items():
    predictions[name] = classifier.predict(X_test)

# Evaluate models
print("\nMODEL EVALUATION RESULTS:")
print("=" * 60)

results = {}
for name, y_pred in predictions.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

    print(f"\n{name.upper()} RESULTS:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

# Compare models
print("\nMODEL COMPARISON:")
print("=" * 40)
for name, metrics in sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True):
    print(f"{name}: Accuracy = {metrics['accuracy']:.4f}")

best_model_name = max(results.items(), key=lambda x: x[1]['accuracy'])[0]
print(f"\nBest performing model: {best_model_name}")

MODEL TRAINING AND EVALUATION
Training set: 3200 articles
Test set: 800 articles
Number of features: 2002

Initializing classifiers...
Training models...
  Random Forest trained
  SVM trained
  Naive Bayes trained

MODEL EVALUATION RESULTS:

RANDOM FOREST RESULTS:
Accuracy: 0.6987
Precision: 0.7187
Recall: 0.6987
F1-Score: 0.7034

Classification Report:
              precision    recall  f1-score   support

     Culture       0.65      0.60      0.63       101
     Economy       0.69      0.67      0.68       108
      Health       0.96      0.82      0.88       109
       Other       0.82      0.72      0.77        93
    Politics       0.68      0.77      0.72       103
    Religion       0.51      0.73      0.60        95
      Sports       0.61      0.65      0.63        97
  Technology       0.82      0.63      0.71        94

    accuracy                           0.70       800
   macro avg       0.72      0.70      0.70       800
weighted avg       0.72      0.70      0.70     

#Prediction Examples

In [ ]:
import random

print("PREDICTION EXAMPLES")
print("=" * 50)

print("\nPredicting categories for ten new/unseen Arabic news articles:")
print("=" * 60)

# Get the best model (SVM)
best_model = classifiers['SVM']

# Select 10 random test articles as "new/unseen" examples
test_indices = random.sample(range(len(y_test)), 10)

print("\nPREDICTION RESULTS:")
print("=" * 60)

for i, idx in enumerate(test_indices, 1):
    actual_category = y_test.iloc[idx]

    # Get the corresponding feature vector
    X_sample = X_test[idx]

    # Make prediction
    predicted_category = best_model.predict(X_sample)[0]

    # Get confidence score (for SVM, use decision function)
    if hasattr(best_model, 'decision_function'):
        confidence_scores = best_model.decision_function(X_sample)
        # Convert to probability-like scores
        confidence = np.max(confidence_scores) - np.min(confidence_scores)
        confidence_percent = min(100, max(0, (confidence + 10) * 5))  # Scale to 0-100
    else:
        confidence_percent = "N/A"

    # Get the original text
    original_text_index = y_test.index[idx]
    article_text = df_balanced.loc[original_text_index, 'text']

    print(f"\nExample {i}:")
    print(f"Article: {article_text[:100]}...")
    print(f"Actual Category: {actual_category}")
    print(f"Predicted Category: {predicted_category}")
    if confidence_percent != "N/A":
        print(f"Confidence Score: {confidence_percent:.1f}%")
    print(f"Result: {'CORRECT' if actual_category == predicted_category else 'WRONG'}")
    print("-" * 50)

# Calculate accuracy on these examples
correct_predictions = sum(1 for i, idx in enumerate(test_indices)
                         if y_test.iloc[idx] == best_model.predict(X_test[idx])[0])

print(f"\nSUMMARY:")
print(f"Correct predictions: {correct_predictions}/10 ({correct_predictions/10*100:.1f}%)")
print(f"Best model: SVM")
print(f"Overall test accuracy: {results['SVM']['accuracy']:.4f}")

PREDICTION EXAMPLES

Predicting categories for ten new/unseen Arabic news articles:

PREDICTION RESULTS:

Example 1:
Article: ريم البارودي تدخل في نوبه بكاء شديده وفاه ميرنا المهندس...
Actual Category: Religion
Predicted Category: Religion
Confidence Score: 55.8%
Result: CORRECT
--------------------------------------------------

Example 2:
Article: ناءب ب اسكان البرلمان يطالب بحصر مشروعات الصرف الصحي المتوقفه في القري...
Actual Category: Politics
Predicted Category: Politics
Confidence Score: 63.6%
Result: CORRECT
--------------------------------------------------

Example 3:
Article: بالصور سيد خطاب يشهد احتفاليه في حب مصر بمنشاه ناصر...
Actual Category: Culture
Predicted Category: Culture
Confidence Score: 54.3%
Result: CORRECT
--------------------------------------------------

Example 4:
Article: حبسه عامين ينتظر طبيب الكركمين اجازه عيد الفطر...
Actual Category: Other
Predicted Category: Technology
Confidence Score: 52.5%
Result: WRONG
---------------------------------------------

#GUI

In [ ]:
import gradio as gr
from collections import Counter

print("INTERACTIVE WEB INTERFACE - ARABIC VERSION")
print("=" * 50)

def predict_news_category(text):
    """Predict category for Arabic news text"""
    if not text.strip():
        return "الرجاء إدخال نص المقال", "0%"

    # Preprocess the input text
    cleaned_text = normalise_arabic(text)
    tokens = tokenize_arabic(text)

    # Vectorize the text
    text_vectorized = vectorizer.transform([cleaned_text])
    text_vectorized_selected = selector.transform(text_vectorized)

    # Calculate handcrafted features
    char_len = len(cleaned_text)
    unique_cnt = len(set(tokens))

    handcrafted = [[char_len, unique_cnt]]
    final_features = hstack([text_vectorized_selected, handcrafted])

    # Make prediction
    prediction = classifiers['SVM'].predict(final_features)[0]

    # Calculate confidence
    confidence_scores = classifiers['SVM'].decision_function(final_features)[0]
    confidence = np.max(confidence_scores)
    confidence_percent = min(100, max(0, (confidence + 2) * 25))

    # Extract keywords
    keyword_tokens = [token for token in tokens if len(token) > 2]
    word_freq = Counter(keyword_tokens)
    keywords = [word for word, count in word_freq.most_common(5)]

    # Prepare Arabic output
    result = f"التصنيف: {prediction}\n"
    result += f"الكلمات المفتاحية: {', '.join(keywords)}\n"
    result += f"ملخص النص: {cleaned_text[:100]}..." if len(cleaned_text) > 100 else f"ملخص النص: {cleaned_text}"

    return result, f"{int(confidence_percent)}%"

# Arabic CSS styling with RTL support and new colors
arabic_css = """
.gradio-container {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', 'Tahoma', 'Segoe UI', sans-serif;
    background-color: #e6f3ff;
}

.contain {
    background: linear-gradient(135deg, #e6f3ff 0%, #d4e7ff 100%);
    border-radius: 12px;
    padding: 25px;
    margin: 15px;
    box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);
    border: 1px solid #c4d9ff;
}

/* Input box styling */
textarea {
    direction: rtl;
    text-align: right;
    background-color: #ffffff;
    border: 2px solid #36634d;
    border-radius: 8px;
    font-size: 16px;
    padding: 15px;
    font-family: 'Arial', 'Tahoma', sans-serif;
    line-height: 1.6;
    color: black !important;
}

textarea:focus {
    border-color: #4a7fff;
    box-shadow: 0 0 0 3px rgba(74, 127, 255, 0.1);
    outline: none;
     color: black !important;
    background-color: #ffffff !important;
}

/* Output box styling */
.contain > div:last-child {
    background-color: #ffffff;
    border: 2px solid #a8c6ff;
    border-radius: 8px;
    padding: 20px;
    direction: rtl;
    text-align: right;
    font-size: 15px;
    line-height: 1.8;
    color: black;
}
.output-text {
    color: black !important;
}
/* Button styling */
button {
    background: #7285a7;
    color: white;
    border-radius: 8px;
    font-weight: 600;
    padding: 14px 32px;
    font-size: 16px;
    border: none;
    margin: 10px 0;
    transition: all 0.3s ease;
    font-family: 'Arial', 'Tahoma', sans-serif;
}

button:hover {
    background: #8c5eba;
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(74, 127, 255, 0.3);
}

/* Label styling */
.label {
    font-weight: 700;
    color: #2d3748;
    font-size: 18px;
    margin-bottom: 12px;
    text-align: right;
    font-family: 'Arial', 'Tahoma', sans-serif;
}

/* Header styling */
h1 {
    color: #2d3748;
    text-align: center;
    margin-bottom: 25px;
    font-size: 2.2em;
    font-weight: 700;
    border-bottom: 3px solid #a8c6ff;
    padding-bottom: 15px;
}

h2 {
    color: #4a5568;
    text-align: right;
    font-size: 1.5em;
    font-weight: 600;
    margin: 20px 0 15px 0;
}

h3 {
    color: #8c5eba;
    text-align: right;
    font-size: 1.3em;
    font-weight: 600;
    margin: 25px 0 15px 0;
    border-right: 4px solid #8c5eba;
    padding-right: 15px;
}

/* Examples styling */
.examples {
    background-color: #deede6;
    border-radius: 8px;
    padding: 18px;
    margin: 15px 0;
    border: 1px solid #a8c6ff;
}

.examples .label {
    color: #4a5568;
    font-size: 16px;
}

/* Confidence label styling */
.confidence-label {
    background-color: #d9c8ea;
    color: black;
    padding: 8px 16px;
    border-radius: 20px;
    font-weight: 600;
    text-align: center;
    margin-top: 10px;
    border: 1px solid #8c5eba;
}

/* Text styling */
.markdown-text {
    line-height: 1.7;
    color: #4a5568;
    text-align: right;
}
.markdown-text, .markdown-text * {
    color: black !important;
}

/* Grid layout */
.gr-row {
    gap: 20px;
}

.gr-column {
    gap: 15px;
}
"""

# Create the Arabic interface
print("Creating Arabic Gradio interface...")

with gr.Blocks(css=arabic_css, theme=gr.themes.Soft(), title="نظام تصنيف المقالات العربية") as demo:
    gr.Markdown("# نظام تصنيف المقالات العربية الآلي")
    gr.Markdown(
        """
    ### كيفية الاستخدام:
    <div style="color: black !important;">
    1. الصق المقال العربي في المربع الأيسر<br>
    2. اضغط على زر 'تصنيف المقال'<br>
    3. شاهد النتائج مع التصنيف والكلمات المفتاحية<br>
    </div>
    """
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("### أدخل نص المقال العربي")
            input_text = gr.Textbox(
                label="النص العربي",
                placeholder="الصق أو اكتب المقال العربي هنا...",
                lines=6,
                max_lines=10
            )
            submit_btn = gr.Button("تصنيف المقال", variant="primary")

        with gr.Column():
            gr.Markdown("### نتائج التصنيف")
            output_text = gr.Textbox(
                label="النتائج",
                lines=6,
                interactive=False
            )
            confidence = gr.Label(
                label="درجة الثقة:",
                value="0%",
                show_label=True
            )

    gr.Markdown("### أمثلة")
    examples = gr.Examples(
            examples=[
                ["تقدم منتخب مصر بثلاثية نظيفة في شباك منتخب غانا في المباراة التي جمعتهما مساء اليوم ضمن منافسات كأس الأمم الأفريقية. وسجل الأهداف محمد صلاح ومحمود حسن ومروان محسن."],
                ["أعلنت وزارة المالية عن موازنة جديدة للدولة تتضمن زيادة في الإنفاق على الصحة والتعليم. كما أشارت إلى خطة لخفض عجز الموازنة خلال العام القادم من خلال تحسين كفاءة الإنفاق."],
                ["كشفت شركة سامسونج عن هاتفها الذكي الجديد الذي يتميز بتقنيات متطورة في الذكاء الاصطناعي. الجهاز الجديد يحتوي على كاميرا بدقة 200 ميجابكسل ومعالج من الجيل الثامن."],
                ["نشرت منظمة الصحة العالمية تقريراً جديداً يحذر من انتشار مرض إنفلونزا الطيور في عدة دول آسيوية. وأوصت باتخاذ إجراءات وقائية مشددة والسيطرة على تداول الدواجن."]
            ],
        inputs=input_text,
        label=""
    )


    # Connect the interface
    submit_btn.click(
        fn=predict_news_category,
        inputs=input_text,
        outputs=[output_text, confidence]
    )

print("Launching Arabic interface...")
demo.launch(share=True)


INTERACTIVE WEB INTERFACE - ARABIC VERSION
Creating Arabic Gradio interface...
Launching Arabic interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbaf3fd37b08dbb683.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
